# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PNRS2006/flyrank-internship-pnrs/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies my Search Intelligence data contract using the FlyRank warehouse.

The lane is **Search Intelligence / content-performance opportunity scoring**. The goal is to use historical information available at a decision moment to identify content items that may need review because their search visibility is weakening.

I use **March 2026** as the mid-panel development and verification month. The final month is treated as a sealed outcome period rather than being used to design the features.


## 1. Unit of analysis + time window

### Contract answers

1. **What one row means:**  
   One row represents one content item for one client on one reporting date in the daily search-performance warehouse.

2. **Which table(s) I will use:**  
   I will use `fact_content_daily_performance` as the main time-series table and `dim_content` for content-level context when needed. The daily fact is the primary source for search-performance features and the outcome.

3. **Time window:**  
   I use **March 2026** as the mid-panel development and verification window. I avoid using the final June 2026 sample as a development window because it represents the natural end/outcome period.

4. **What I will predict/rank:**  
   I will rank content items by their risk of a future search-impression decline. The working proxy label is whether impressions decline by more than 20% in the outcome window.

5. **One deliberate exclusion:**  
   I deliberately exclude `trend_direction` and `trend_pct` from the model features because they are derived from outcome/trend information and could leak label information into the model.


In [ ]:
# This cell verifies that the notebook can access the required Python packages.
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os
import getpass
import duckdb
import pandas as pd

print("Python environment ready.")


In [ ]:
# Read the Hugging Face token from the environment or Colab Secret.
# Never hard-code the token in this notebook.

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face READ token (hf_...): "
    )

print("Hugging Face token loaded:", bool(HF_TOKEN))


In [ ]:
# Connect DuckDB to the FlyRank warehouse.

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Warehouse connection ready.")
print("Available warehouse tables:")
        
      for name in TABLES:
        print(" -", name)


## 2. Fields: feature / label / context / excluded

### Feature fields

The five initial features are:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`

These represent search visibility, search traffic, and engagement signals that can be observed before the future outcome is evaluated.

### Label

The working label is:

**`is_declining_label` = 1 when future search impressions decline by more than 20%; otherwise 0.**

This is a practical proxy for search-performance decline, not a claim about Google's ranking algorithm.

### Context fields

- `client_hash_id`
- `content_hash_id`
- `report_date`

These identify the client, content item, and observation date and are used for grouping, filtering, and validation rather than as model features.

### Excluded fields

- `trend_direction`
- `trend_pct`

These are excluded because they directly describe the trend/outcome used to construct the decline label.

I also exclude pseudonymous IDs such as `client_hash_id` and `content_hash_id` from model features because IDs are useful for grouping and joins but should not be treated as predictive variables.


In [ ]:
# Inspect the available columns before running the verification queries.

schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1"
).df()

print("Columns in fact_content_daily_performance:")
print(schema[["column_name", "column_type"]].to_string(index=False))


## 3. Verify it with exactly three queries

The following three queries verify the contract on the **March 2026** development slice.

1. Grain
2. Row count and date span
3. Availability using `IS TRUE`


### Query 1 — Grain verification

This query checks whether one warehouse row represents one client × content item × reporting date.

In [ ]:
# QUERY 1 — Grain verification

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            client_hash_id || '|' ||
            content_hash_id || '|' ||
            CAST(report_date AS VARCHAR)
        ) AS distinct_client_content_date,
        COUNT(*) - COUNT(DISTINCT
            client_hash_id || '|' ||
            content_hash_id || '|' ||
            CAST(report_date AS VARCHAR)
        ) AS duplicate_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

grain_check


### Query 1 interpretation

The expected result is `duplicate_rows = 0`.

That verifies that the March 2026 daily fact behaves as:

**one row = one client × one content item × one reporting date.**

### Query 2 — Slice row count and date span

This query measures the size of the March 2026 development slice and the dates represented in it.

In [ ]:
# QUERY 2 — March 2026 row count and date span

march_counts = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

march_counts


### Query 2 interpretation

This output records the measured size of the March 2026 slice and its observed date range.

I use this mid-panel month for development rather than the final June 2026 sample.

### Query 3 — Availability using `IS TRUE`

This query checks how many March rows have explicitly available GSC and GA4 data.

In [ ]:
# QUERY 3 — Availability check
# The availability flags are checked explicitly with IS TRUE.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check


### Query 3 interpretation

Availability is measured explicitly with `IS TRUE`.

I do not treat a zero-filled GA4 value as equivalent to unavailable data. I require the relevant availability flag to be explicitly TRUE before treating the source as available.

## Five-feature frame

I now build a small March 2026 feature frame using only five features.

The feature frame is deliberately kept small so that every feature can be inspected for availability and leakage.

In [ ]:
# Build the five-feature frame from March 2026.

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()


### Feature availability — knowable at the decision moment because...

- **`gsc_impressions`** — Knowable at the decision moment because these are search impressions already observed in the historical window before the future outcome is evaluated.

- **`gsc_clicks`** — Knowable at the decision moment because these are search clicks already recorded by GSC in the historical observation window.

- **`gsc_avg_position`** — Knowable at the decision moment because it summarizes the search position already observed during the historical window.

- **`ga4_sessions`** — Knowable at the decision moment because these sessions have already occurred and are available from GA4 history.

- **`ga4_engaged_sessions`** — Knowable at the decision moment because these engaged sessions are historical GA4 observations available before the future outcome window.

## Deliberate leakage experiment

I intentionally create one label-derived feature to demonstrate the leakage trap.

The deliberately leaked feature is `leakage_feature = is_declining_label`.

This feature would never be available when making the actual decision because it is derived directly from the outcome.

The purpose of this experiment is to show why a suspiciously perfect score should not automatically be trusted.

In [ ]:
# Construct a simple outcome proxy for the leakage demonstration.
# The label compares an early March period with a later March period.

label_data = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            SUM(gsc_impressions) AS impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-03-15'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id,
            report_date
    ),
    periods AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(
                CASE
                    WHEN report_date >= DATE '2026-03-01'
                     AND report_date < DATE '2026-03-08'
                    THEN impressions
                    ELSE 0
                END
            ) AS first_7d,
            SUM(
                CASE
                    WHEN report_date >= DATE '2026-03-08'
                     AND report_date < DATE '2026-03-15'
                    THEN impressions
                    ELSE 0
                END
            ) AS second_7d
        FROM daily
        GROUP BY
            client_hash_id,
            content_hash_id
        HAVING first_7d > 0
    )
    SELECT
        client_hash_id,
        content_hash_id,
        first_7d,
        second_7d,
        CASE
            WHEN second_7d < 0.8 * first_7d THEN 1
            ELSE 0
        END AS is_declining_label
    FROM periods
""").df()

print("Rows available for leakage demonstration:", len(label_data))
label_data.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Join the historical feature frame to the outcome proxy.

demo = feature_frame.merge(
    label_data[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

demo = demo.dropna(
    subset=honest_features + ["is_declining_label"]
).copy()

print("Rows in modelling demonstration:", len(demo))
print("Positive labels:", int(demo["is_declining_label"].sum()))
print("Negative labels:", int((demo["is_declining_label"] == 0).sum()))


In [ ]:
# Honest baseline using only historical features.

X = demo[honest_features]
y = demo["is_declining_label"]

if y.nunique() < 2:
    raise ValueError(
        "The March slice produced only one label class. "
        "The leakage demonstration cannot be evaluated with a classifier."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)
honest_pred = honest_model.predict(X_test)
honest_accuracy = accuracy_score(y_test, honest_pred)

print(f"Honest feature accuracy: {honest_accuracy:.3f}")


In [ ]:
# DELIBERATE LEAKAGE
#
# Copy the label itself into the feature matrix.
# This is intentionally wrong and exists only to demonstrate leakage.

leaky = demo.copy()
leaky["leakage_feature"] = leaky["is_declining_label"]

leaky_features = honest_features + ["leakage_feature"]
        
        X_leaky = leaky[leaky_features]
        y_leaky = leaky["is_declining_label"]

        X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
            X_leaky,
            y_leaky,
            test_size=0.25,
            random_state=42,
            stratify=y_leaky
        )

        leaky_model = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )

        leaky_model.fit(X_train_l, y_train_l)
        leaky_pred = leaky_model.predict(X_test_l)
        leaky_accuracy = accuracy_score(y_test_l, leaky_pred)

        print(f"Honest accuracy: {honest_accuracy:.3f}")
        print(f"Leaky accuracy:  {leaky_accuracy:.3f}")
        print(f"Accuracy jump:   {leaky_accuracy - honest_accuracy:.3f}")


### Leakage result

The deliberately leaked feature contains the target itself, so the model can use outcome information that would not exist at the decision moment.

The resulting score is therefore invalid evidence.

This experiment demonstrates the central leakage lesson:

> A feature can be mathematically predictive while being operationally impossible to know when the decision must be made.

The leaky feature is removed before keeping the final feature set.

In [ ]:
# Remove the deliberate leakage feature.
# The final feature set contains only the five historical features.
        
        final_feature_cols = honest_features.copy()

        final_features = demo[
            [
                "client_hash_id",
                "content_hash_id",
                "is_declining_label"
            ] + final_feature_cols
        ].copy()

        print("Final feature columns:")
        for feature in final_feature_cols:
            print(" -", feature)

        print("\nLeakage feature present:", "leakage_feature" in final_features.columns)
        print("Final feature frame shape:", final_features.shape)
        final_features.head()


## 4. Data limits

### Named limitation: unbalanced history

The warehouse is an **unbalanced panel**: different clients have different amounts of historical data.

Therefore, a March 2026 slice does not necessarily represent the same amount of historical context for every client.

Another limitation is that GSC and GA4 availability differs across clients and dates. Rows without an explicit availability flag of TRUE are not treated as equivalent to observed zero activity.

The query-level table also has a fixed 90-day window, so recent-window metrics can overlap an outcome period depending on how the prediction window is defined. I therefore do not use overlapping outcome-period values as features in this contract.

Finally, the decline label is a **decision-support proxy**, not proof that a page is objectively underperforming or that a particular search-engine algorithm caused the change.

In [ ]:
# Compact record of the final data contract.

contract_summary = {
    "lane": "Search Intelligence / content-performance opportunity scoring",
    "unit_of_analysis": "one client-content item per reporting date",
    "development_month": "2026-03",
    "label_proxy": "future impressions decline > 20%",
    "feature_count": len(final_feature_cols),
    "features": final_feature_cols,
    "excluded": [
        "trend_direction",
        "trend_pct",
        "client_hash_id",
        "content_hash_id"
    ],
    "availability_rule": "gsc_data_available IS TRUE AND ga4_data_available IS TRUE"
}

for key, value in contract_summary.items():
    print(f"{key}: {value}")


## Self-check

- [x] The notebook defines the Search Intelligence lane.
- [x] One row is defined as one client × content × reporting date.
- [x] March 2026 is used as the mid-panel development/verification month.
- [x] The warehouse tables are explicitly identified.
- [x] The prediction target is defined as a future search-impression decline proxy.
- [x] Exactly three verification queries are included.
- [x] Query 1 verifies the grain.
- [x] Query 2 reports the March 2026 row count and date span.
- [x] Query 3 checks availability using `IS TRUE`.
- [x] A five-feature frame is constructed.
- [x] Each of the five features has an "available at the decision moment" explanation.
- [x] One label-derived leakage feature is deliberately introduced.
- [x] The leaky score is compared with the honest score.
- [x] The leakage feature is removed from the final feature set.
- [x] One named limitation is documented.
- [x] No client names, URLs, private queries, or credentials are included.
- [x] The notebook should be run from top to bottom before committing.
- [x] The completed notebook belongs under `work/notebooks/w03_data_contract.ipynb`.